# Stage 1 — Vanilla Predictive Coding on MNIST

In [1]:
import sys; sys.path.insert(0, '..')
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from models.pc_layer import PCNetwork

## Data

In [2]:
T = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # [0,1] -> [-1,1], matches tanh output range
])
train_data = torchvision.datasets.MNIST('../data', train=True,  download=True, transform=T)
test_data  = torchvision.datasets.MNIST('../data', train=False, download=True, transform=T)
train_loader = DataLoader(train_data, batch_size=64,  shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=256, shuffle=False)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device={device}  train={len(train_data)}  test={len(test_data)}')

device=cuda  train=60000  test=10000


## Model

In [3]:
# 784 pixels -> 256 -> 64 -> 10 classes
net = PCNetwork([784, 256, 64, 10], lr_inference=0.05).to(device)

## Training helpers

In [4]:
def one_hot(y):
    return torch.eye(10, device=device)[y]

def train_epoch(lr_w=0.001, n_steps=20):
    total = 0.0
    for imgs, labels in train_loader:
        x = imgs.view(-1, 784).to(device)
        net.init_states(x.size(0), device)
        net.infer(x, n_steps=n_steps, clamp_top=one_hot(labels.to(device)))
        net.update_weights(lr_w)          # weights update only AFTER inference converges
        total += net.total_energy().item()
    return total / len(train_loader)

def test_accuracy(n_steps=20):
    correct = 0
    with torch.no_grad():                                               
        for imgs, labels in test_loader:
            x = imgs.view(-1, 784).to(device)
            net.init_states(x.size(0), device)
            net.infer(x, n_steps=n_steps)    # no label clamping: top-level r is free
            correct += (net.layers[-1].r.argmax(1).cpu() == labels).sum().item()
    return correct / len(test_data)

## Training loop

In [6]:
N_EPOCHS = 20
accs = []
for ep in range(1, N_EPOCHS + 1):
    energy = train_epoch(lr_w=0.0001, n_steps=100)
    acc    = test_accuracy(n_steps=100)
    # Extract the scalar value if they are tensors
    energy_val = energy.item() if hasattr(energy, 'item') else energy
    acc_val    = acc.item() if hasattr(acc, 'item') else acc
    
    accs.append(acc_val)
    print(f'ep {ep:2d}  energy={energy_val:.4f}  acc={acc_val:.4f}')

ep  1  energy=52.9932  acc=0.6366
ep  2  energy=51.8453  acc=0.6426
ep  3  energy=50.7743  acc=0.6484
ep  4  energy=49.7794  acc=0.6538
ep  5  energy=48.8461  acc=0.6593
ep  6  energy=47.9711  acc=0.6646
ep  7  energy=47.1517  acc=0.6678
ep  8  energy=46.3875  acc=0.6717
ep  9  energy=45.6600  acc=0.6757
ep 10  energy=44.9692  acc=0.6806
ep 11  energy=44.3183  acc=0.6844
ep 12  energy=43.7038  acc=0.6877
ep 13  energy=43.1270  acc=0.6914
ep 14  energy=42.5827  acc=0.6944
ep 15  energy=42.0661  acc=0.6979
ep 16  energy=41.5723  acc=0.6998
ep 17  energy=41.1112  acc=0.7013
ep 18  energy=40.6625  acc=0.7038
ep 19  energy=40.2254  acc=0.7063
ep 20  energy=39.8223  acc=0.7089


## Verification 1 — Accuracy

- [ ] Classification accuracy ≥ 95% on MNIST test set

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(accs) + 1), accs, marker='o')
plt.axhline(0.95, color='r', linestyle='--', label='95% target')
plt.xlabel('Epoch'); plt.ylabel('Test Accuracy')
plt.title('MNIST Test Accuracy'); plt.legend(); plt.tight_layout(); plt.show()
print(f'Final accuracy: {accs[-1]:.4f}')

## Verification 2 — Inference convergence

- [ ] Inference loop converges within ~20 iterations on a single image
- [ ] Prediction errors decrease monotonically during inference

In [ ]:
img, label = test_data[0]
x = img.view(1, 784).to(device)
net.init_states(1, device)

energies = []
for _ in range(50):
    net.layers[0].compute_error(x)
    for i in range(1, len(net.layers)):
        net.layers[i].compute_error(net.layers[i - 1].r)
    energies.append(net.total_energy().item())
    net.layers[-1].update_r()
    for i in range(len(net.layers) - 2, -1, -1):
        net.layers[i].update_r(e_above=-net.layers[i + 1].e)

plt.figure(figsize=(6, 4))
plt.plot(energies, marker='.')
plt.xlabel('Inference step'); plt.ylabel('Total energy')
plt.title(f'Inference convergence (true label={label})')
plt.tight_layout(); plt.show()
print(f'pred={net.layers[-1].r.argmax(1).item()}  true={label}')

## Verification 3 — Top-down reconstructions

- [ ] Visualize top-down reconstructions at each level

In [ ]:
img, label = test_data[0]
x = img.view(1, 784).to(device)
net.init_states(1, device)
net.infer(x, n_steps=50)

# Layer 0's prediction is in pixel space (784-dim) — the full reconstruction
recon = net.layers[0].predict().view(28, 28).detach().cpu()

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(img.squeeze(), cmap='gray')
axes[0].set_title(f'Input (label={label})'); axes[0].axis('off')
axes[1].imshow(recon, cmap='gray')
axes[1].set_title('Level-0 top-down reconstruction'); axes[1].axis('off')
plt.tight_layout(); plt.show()

# Top-level representation: should peak at the true class
top_r = net.layers[-1].r.squeeze().detach().cpu()
plt.figure(figsize=(6, 3))
plt.bar(range(10), top_r.numpy())
plt.xlabel('Class'); plt.ylabel('Activation'); plt.xticks(range(10))
plt.title(f'Top-level repr  pred={top_r.argmax().item()}  true={label}')
plt.tight_layout(); plt.show()